In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # EfficientNetV2 ve CvT gibi modeller için kritik

from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau # LR Scheduler'lar için

# Uyarıları gizlemek
warnings.filterwarnings("ignore")

# --- GLOBAL AYARLAR VE TEKRARLANABİLİRLİK ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Gerekli tüm kütüphaneler ve global ayarlar yüklendi.")

In [ ]:

# --- KONFİGÜRASYON (CvT - Convolutional Vision Transformer) ---
# CvT-13 Modeli (timm kodu: cvt_13) - Denge ve performans için tercih edilen varyant.
MODEL_NAME = 'cvt_13'

# Deney Adı (Takip kolaylığı için)
EXPERIMENT_NAME = "CvT_13_Augmented_AdamW_Run1"

# Hiperparametreler
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.00005
NUM_CLASSES = 8
DROPOUT_RATE = 0.1
# CvT için KRİTİK: L2 düzenlileştirme (Weight Decay) daha yüksek ayarlanır (AdamW ile kullanılır).
WEIGHT_DECAY = 0.05

# --- DOSYA YOLLARI ---
DATA_DIR = "../../data/prepared_data_aug"

# Sonuçların kaydedileceği yer
OUTPUT_DIR = f"../../models/pytorch/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cihaz Kontrolü
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Kayıt Yeri: {OUTPUT_DIR}")

In [ ]:
# ImageNet Normalize Değerleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri Dönüşümleri
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri Oluştur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderları Oluştur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=='train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"Sınıflar: {class_names}")
print(f"Eğitim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")

In [ ]:
def create_model(model_name: str, num_classes: int, dropout_rate: float, device: str):
    """
    Belirtilen model adıyla (timm) modeli oluşturur, ağırlıklarını yükler ve cihaz üzerine taşır.
    """
    print(f"\nModel indiriliyor: {model_name}...")

    # pretrained=True ile ImageNet ağırlıklarını alıyoruz
    try:
        model = timm.create_model(model_name,
                                  pretrained=True,
                                  num_classes=num_classes,
                                  drop_rate=dropout_rate)
    except Exception as e:
        print(f"Hata: Model '{model_name}' yüklenemedi. Kontrol edin. Hata: {e}")
        return None

    model = model.to(device)
    return model

# --- KURULUM ---
# MODEL_NAME, NUM_CLASSES, DROPOUT_RATE ve DEVICE değişkenlerinin konfigürasyon bloğundan geldiği varsayılır.
model = create_model(MODEL_NAME, NUM_CLASSES, DROPOUT_RATE, DEVICE)

# Kayıp Fonksiyonu
criterion = nn.CrossEntropyLoss()

# Optimizer: CvT için kritik olan AdamW kullanılıyor.
# WEIGHT_DECAY, daha önce tanımladığınız CvT konfigürasyon bloğundan (0.05) geliyor.
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning Rate Scheduler: CosineAnnealingLR, Transformer tabanlı modeller için en iyi performansı sağlar.
# T_max (EPOCHS) konfigürasyondan gelmelidir.
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("CvT Model GPU'ya yüklendi ve eğitime hazır.")

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    # DATALOADERS ve DATASET_SIZES'ın global olarak tanımlandığını varsayıyoruz.

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch Döngüsü
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Epoch Sonucu Hesaplama
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Geçmişi Kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # --- SCHEDULER VE MODEL KAYDI ---

            if phase == 'train':
                # CosineAnnealingLR (CvT için önerilen) her eğitim epoch'undan sonra adım atar.
                scheduler.step()
                current_lr = optimizer.param_groups[0]['lr']
                print(f"Current LR: {current_lr:.6f}")

            if phase == 'val':
                # En iyi modeli kaydet (Validasyon başarısına göre)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    save_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
                    torch.save(model.state_dict(), save_path)
                    print(f"En İyi Model! ({best_acc:.4f}) -> Kaydedildi ve Yolu: {save_path}")

    time_elapsed = time.time() - since
    print(f'\nEğitim Tamamlandı: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En İyi Validasyon Doğruluğu: {best_acc:.4f}')

    # En iyi ağırlıkları geri yükle
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    return model, history

In [ ]:
# --- BAŞLAT ---
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
plt.figure(figsize=(14, 5))

# Doğruluk Grafiği
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title(f'{MODEL_NAME} Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Kayıp Grafiği
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title(f'{MODEL_NAME} Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Kaydet ve Göster
plt.savefig(os.path.join(OUTPUT_DIR, 'training_graph.png'))
plt.show()
print(f"Grafikler kaydedildi: {os.path.join(OUTPUT_DIR, 'training_graph.png')}")

In [ ]:
print("\nTEST SETİ DEĞERLENDİRMESİ")

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 1. Classification Report (F1, Recall, Precision)
print("\nSınıflandırma Raporu:")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'))
plt.show()